**SOC612: Data Analytics for the Social Sciences**

**Date: September 22, 2026**

**Author: R. Duerr**

**Lecture 3: Descriptive and exploratory data analysis**

To apply some techniques for data descriptions and explorations, we use the National Health Interview Survey NHIS (2020). It contains many health-, medical-, and lifestyle-related variables. We explore the subjective health status of the sample participants, as well as some of their demographics, and lifestyle-related choices.

In a first step, we tidy the dataset, and subset our variables of interest to prepare it for our further analytic purposes.


In [ ]:
import pandas as pd
import numpy as np

# Read the CSV file
data = pd.read_csv("nhis-20-a.csv")

# Select and rename columns
data = data[['PHSTAT_A', 'ECIGEV_A', 'SMKEV_A', 'SLPHOURS_A', 'DRKLIFE_A',
              'AGEP_A', 'SEX_A', 'FAMINCTC_A', 'EDUC_A']].rename(columns={
    'PHSTAT_A': 'health_status',
    'ECIGEV_A': 'lfstl_vap',
    'SMKEV_A': 'lfstl_smo',
    'SLPHOURS_A': 'lfstl_sle',
    'DRKLIFE_A': 'lfstl_alc',
    'AGEP_A': 'age',
    'SEX_A': 'gender',
    'FAMINCTC_A': 'family_income',
    'EDUC_A': 'education'
})

# Recoding health_status
health_status_map = {'1': 'excellent', '2': 'very_good', '3':
                     'good', '4': 'fair', '5': 'poor'}
data['health_status'] = data['health_status'].astype(str).map(health_status_map)
data['health_status'] = pd.Categorical(
    data['health_status'],
    categories=['poor', 'fair', 'good', 'very_good', 'excellent'],
    ordered=True
)

# Recoding gender
gender_map = {'1': 'male', '2': 'female'}
data['gender'] = data['gender'].astype(str).map(gender_map)
data['gender'] = pd.Categorical(data['gender'],
                                categories=['male', 'female'], ordered=True)

# Recoding education
conditions = [
    data['education'].isin([0, 1]),
    data['education'].isin([2, 3, 4, 5]),
    data['education'].isin([6, 7]),
    data['education'].isin([8, 9, 10, 11])
]
choices = ['primary_or_below', 'secondary', 'post_secondary', 'tertiary']
data['education'] = np.select(conditions, choices, default=np.nan)
data['education'] = pd.Categorical(
    data['education'],
    categories=['primary_or_below', 'secondary', 'post_secondary', 'tertiary'],
    ordered=True
)

# Recoding lifestyle variables
for col in ['lfstl_vap', 'lfstl_smo', 'lfstl_alc']:
    var_map = {'1': 'yes', '2': 'no'}
    data[col] = data[col].astype(str).map(var_map)
    data[col] = pd.Categorical(data[col], categories=['yes', 'no'], ordered=True)

# Cleaning lfstl_sle: Set values > 24 or < 1 to NA
data['lfstl_sle'] = np.where(
    (data['lfstl_sle'] > 24) | (data['lfstl_sle'] < 1),np.nan, data['lfstl_sle']
)
data['lfstl_sle'] = pd.to_numeric(data['lfstl_sle'])

# Remove rows with NA
data = data.dropna()

Our dataset "data" contains 30,328 complete observations of 9 variables. We can take a first look at it by using .info() or .head().

In a further step, we can look at the aggregate measures of the dataset, using, for example, .describe().

In [ ]:
# glimpse (structure of the data)
print(data.info())

# head (first 10 observations)
print(data.head(10))

# summary (descriptive statistics for all columns)
print(data.describe(include='all'))

# For a more detailed summary, use pandas_profiling or:
numeric_cols = data.select_dtypes(include=['number']).columns
cat_cols = data.select_dtypes(include=['object', 'category']).columns

print("\n--- Numeric Summary ---")
print(data[numeric_cols].describe())

print("\n--- Categorical Summary ---")
for col in cat_cols:
    print(f"\n{col}:")
    print(data[col].value_counts(dropna=False))

The amount of information we get from categorical and numerical variables is different. We can also compute these measures individually, if we want to extract a specific value, or change an existing one. Functions for common point estimates like mean, median, or sd, among others, are built-in functions, while for others like the IQR-function, a package is sometimes necessary.



In [ ]:
import numpy as np
from scipy.stats import iqr, skew

# Calculate statistics for numeric variables
age_stats = [
    round(data['age'].mean(), 3),
    round(data['age'].std(), 3),
    round(iqr(data['age']), 3),
    round(skew(data['age']), 3),
    round(data['age'].sum(), 3)
]

sleep_stats = [
    round(data['lfstl_sle'].mean(), 3),
    round(data['lfstl_sle'].std(), 3),
    round(iqr(data['lfstl_sle']), 3),
    round(skew(data['lfstl_sle']), 3),
    round(data['lfstl_sle'].sum(), 3)
]

inc_stats = [
    round(data['family_income'].mean(), 3),
    round(data['family_income'].std(), 3),
    round(iqr(data['family_income']), 3),
    round(skew(data['family_income']), 3),
    round(data['family_income'].sum(), 3)
]

# Create a DataFrame for the measures
measures_num = pd.DataFrame({
    'variable': ['mean', 'sd', 'iqr', 'skewness', 'sum'],
    'age': age_stats,
    'sleep': sleep_stats,
    'family_income': inc_stats
})

print(measures_num)

# Summary table for categorical variables (like table1)
cat_vars = ['health_status', 'lfstl_vap', 'lfstl_smo', 'lfstl_alc', 'gender', 'education']
for var in cat_vars:
    print(f"\n{var}:")
    print(data[var].value_counts(dropna=False, normalize=True))

To get a more intuitive understanding of the distribution of our data, it is useful to visually explore the data. There are several ways to go forward here:

For the metric variable (family_income), univariate distributions (plots 1-6):

- Histograms: they show the frequency distribution of a variable by dividing it into bins. They help visualize the shape, spread, and skewness of the data.

- Density plots: they show the probability density function of a variable, smoothing the distribution to highlight the overall shape and peaks. Contrary to histograms, they show a continuous estimate.

- Boxplots: they show the median, quartiles, and potential outliers of a variable. They are ideal for comparing distributions across groups.

For metric variables, bivariate distributions (plots 7-12):

- Histograms, density plots, and boxplots can also be help visualize bivariate (or multivariate) distributions. We plot family_income by gender.

For categorical variables, univariate distributions (plots 13-14):

- Bar plots: bar plots (or bar charts) look similar to histograms, but they visualize the number of cases per category. In this case, we visualize the distribution of the variable health_status.

For categorical variables, bivariate distributions (plots 15-16):

- Bar plots can also be used to show bivariate distributions. We visualize health_status by gender.

There is no clear answer to which one you should use, as all of them have their pro's and con's. However, think about who your audience is and what you want to demonstrate with the plot. Usually a plot is good, when the plot is informative and the main message can be seen very quickly, when the plot aesthetics are simple and not distracting, when there is no overplotting, when the plot elements are not confusing, and when the colors allow you to clearly distinguish the plot elements.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

# 1: Histogram
plt.hist(data['family_income'], color='blue', edgecolor='black', linewidth=2)
plt.title("Histogram")
plt.xlabel("Fam. Inc.")
plt.ylabel("Freq.")
plt.show()

# 2: Density Plot
density = gaussian_kde(data['family_income'])
x = np.linspace(min(data['family_income']), max(data['family_income']), 1000)
plt.plot(x, density(x), color='blue', linewidth=2)
plt.title("Density")
plt.xlabel("Fam. Inc.")
plt.ylabel("Density")
plt.show()

# 3: Boxplot
plt.boxplot(data['family_income'], patch_artist=True,
            boxprops=dict(facecolor='lightblue', color='black'),
            whiskerprops=dict(color='black'),
            capprops=dict(color='black'),
            medianprops=dict(color='black'))
plt.title("Boxplot")
plt.ylabel("Fam. Inc.")
plt.show()

---

# 4: Histogram (seaborn)
sns.histplot(data=data, x='family_income', bins=20,
             color='lightblue', edgecolor='black')
plt.title("Histogram")
plt.xlabel("Fam. Inc.")
plt.ylabel("Freq.")
plt.show()

# 5: Density Plot (seaborn)
sns.kdeplot(data=data, x='family_income', fill='blue', alpha=0.5)
plt.title("Density")
plt.xlabel("Fam. Inc.")
plt.ylabel("Density")
plt.show()

# 6: Boxplot (seaborn)
sns.boxplot(data=data, y='family_income', color='lightblue', linewidth=1)
plt.title("Boxplot")
plt.ylabel("Fam. Inc.")
plt.show()

# --- Separated by Gender ---
# Extract male and female income
male_income = data[data['gender'] == 'male']['family_income']
female_income = data[data['gender'] == 'female']['family_income']

# 7: Histogram by Gender
plt.hist(male_income, bins=20, color=plt.cm.Blues(alpha=0.7),
         edgecolor='black', label='Male')
plt.hist(female_income, bins=20, color=plt.cm.Pink(alpha=0.5),
         edgecolor='black', label='Female')
plt.title("Histogram of Family Income by Gender")
plt.xlabel("Fam. Inc.")
plt.ylabel("Freq.")
plt.xlim(min(male_income.min(), female_income.min()),
         max(male_income.max(), female_income.max()))
plt.ylim(0, 1800)
plt.legend(loc='upper right')
plt.show()

# 8: Density Plot by Gender
density_male = gaussian_kde(male_income)
density_female = gaussian_kde(female_income)
x = np.linspace(min(male_income.min(), female_income.min()),
                max(male_income.max(), female_income.max()), 1000)
plt.plot(x, density_male(x), color='blue', linewidth=2, label='Male')
plt.plot(x, density_female(x), color='red', linewidth=2, label='Female')
plt.title("Density Plot of Fam. Inc. by Gender")
plt.xlabel("Family Income")
plt.ylabel("Dens.")
plt.xlim(min(male_income.min(), female_income.min()),
         max(male_income.max(), female_income.max()))
plt.ylim(0, max(density_male(x).max(), density_female(x).max()))
plt.legend(loc='upper right')
plt.show()

# 9: Boxplot by Gender
gender_data = [data[data['gender'] == g]['family_income'] for g in data['gender'].unique()]
plt.boxplot(gender_data, patch_artist=True,
            boxprops=dict(facecolor=['blue', 'red'], color='black'),
            whiskerprops=dict(color='black'),
            capprops=dict(color='black'),
            medianprops=dict(color='black'))
plt.title("Boxplot of Fam. Inc. by Gender")
plt.ylabel("Family Income")
plt.xticks(range(1, len(data['gender'].unique()) + 1), data['gender'].unique())
plt.show()

# 10: Histogram by Gender (seaborn)
sns.histplot(data=data, x='family_income', hue='gender', bins=20,
             alpha=0.3, multiple='layer')
plt.title("Histogram of Fam. Inc. by Gender")
plt.xlabel("Family Income")
plt.ylabel("Freq.")
plt.show()

# 11: Density Plot by Gender (seaborn)
sns.kdeplot(data=data, x='family_income', hue='gender', fill=True,
            alpha=0.5, common_norm=False)
plt.title("Density Plot of Fam. Inc. by Gender")
plt.xlabel("Family Income")
plt.ylabel("Dens.")
plt.show()

# 12: Boxplot by Gender (seaborn)
sns.boxplot(data=data, x='gender', y='family_income', hue='gender',
            palette=['lightblue', 'lightpink'])
plt.title("Boxplot of Fam. Inc. by Gender")
plt.xlabel("Gender")
plt.ylabel("Family Income")
plt.show()

# --- Bar Charts for Categorical Variables ---
# 13: health_status
health_counts = data['health_status'].value_counts().sort_index()
plt.bar(health_counts.index.astype(str), health_counts.values,
        color='lightblue', edgecolor='black')
plt.title("Bar Chart of Health Status")
plt.xlabel("Health Status")
plt.ylabel("Freq.")
plt.show()

# 14: health_status (seaborn)
sns.countplot(data=data, x='health_status', color='lightblue', edgecolor='black')
plt.title("Bar Chart of Health Status")
plt.xlabel("Health Status")
plt.ylabel("Freq.")
plt.show()

# --- Separated by Gender ---
# 15: health_status by Gender
ct = pd.crosstab(data['gender'], data['health_status'])
ct.plot(kind='bar', color=['lightblue', 'lightpink'], edgecolor='black')
plt.title("Bar Chart of Health Status by Gender")
plt.xlabel("Health Status")
plt.ylabel("Freq.")
plt.legend(title="Gender", loc='upper right')
plt.xticks(rotation=45)
plt.show()

# 16: health_status by Gender (seaborn)
sns.countplot(data=data, x='health_status', hue='gender',
              palette=['lightblue', 'lightpink'])
plt.title("Bar Chart of Health Status by Gender")
plt.xlabel("Health Status")
plt.ylabel("Freq.")
plt.legend(title="Gender")
plt.show()

After plotting the distributions, we can go a step further a explore how strong some variables are correlated.

- 1: We can start by calculating some point estimates (mean, medican etc.) again, this time by another variable. In the first example, we look at family_income by health_status.

- 2: The .corr() function in Python calculates the correlation coefficient (by default Pearson) between numeric variables. It shows the strength and direction of their linear relationship. We create a plot using age, income, health_status, and education. .corr() only creates a numeric output, we can use, for example, sns.heatmap() to visualize the correlations.

- 3: After investigating the correlation, it's useful to look at the distribution of the individual data points, with one variable on the x-axis, and the other on the y-axis. This helps to detect patterns and clusters, and outliers.

- 4: While scatterplots are a popular choice, they sometimes have an overplotting problems, meaning that too many data points are at the same position. This is especially the case, when the range of the variables is small. One alternative is to use jitterplots instead of scatterplots. An more alternative approach is the use of heatmaps, which show the concentration of data points in smaller tiles.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- 1: Grouped Summaries ---
data_grouped = data.groupby('health_status')['family_income'].agg(
    mean_fam_inc='mean',
    median_fam_inc='median',
    max_fam_inc='max',
    min_fam_inc='min',
    count='count'
).reset_index()

print(data_grouped)

# --- 2: Correlation Table and Plots ---
# Select columns and convert factors to numeric codes
data2 = data[['age', 'family_income', 'health_status', 'education']].copy()
data2['health_status'] = pd.to_numeric(data2['health_status'].cat.codes)
data2['education'] = pd.to_numeric(data2['education'].cat.codes)

# Correlation table
corr_table = data2.corr()

# Corrplot (seaborn)
plt.figure(figsize=(8, 6))
sns.heatmap(corr_table, annot=True, fmt=".2f", cmap='coolwarm', center=0, square=True)
plt.title("Correlation Matrix")
plt.show()

# --- 3: Scatterplots ---
# Scatterplot 1: Health Status vs. Family Income
sns.scatterplot(
    data=data,
    x='family_income',
    y=data['health_status'].cat.codes,
    color='blue',
    s=3
)
plt.title("Scatterplot of Health Status vs. Family Income")
plt.xlabel("Fam. Inc.")
plt.ylabel("Health Status")
plt.show()

# Scatterplot 2: Age vs. Family Income
sns.scatterplot(
    data=data,
    x='family_income',
    y='age',
    color='blue',
    s=3
)
plt.title("Scatterplot of Age vs. Family Income")
plt.xlabel("Fam. Inc.")
plt.ylabel("Age")
plt.show()

# --- 4: Heatmaps ---
# Heatmap 1: Family Income vs. Health Status
plt.hist2d(
    data['family_income'],
    pd.to_numeric(data['health_status'].cat.codes),
    bins=20,
    cmap='Reds',
    cmin=0
)
plt.colorbar()
plt.title("Heatmap of Family Income vs. Health Status")
plt.xlabel("Fam. Inc.")
plt.ylabel("Health Status")
plt.show()

# Heatmap 2: Age vs. Family Income
plt.hist2d(
    data['age'],
    data['family_income'],
    bins=20,
    cmap='Reds',
    cmin=0
)
plt.colorbar()
plt.title("Heatmap of Age vs. Family Income")
plt.xlabel("Fam. Inc.")
plt.ylabel("Age")
plt.show()

We should also assume that there might be outliers in our data that can distort the main message of our analysis. There are different techniques to recognize outliers. The decision of what to do with outliers, should be given careful consideration. Whether we decide to remove them, to keep them, to recode them, or any other treatment, we should not forget that they are also legimitate observations. We should therefore always be fully transparent about what and why we do something to them, and what consequences this could have.

A common way is also to do further inferential analysis once with a full dataset, and once with an outlier-free dataset.

- 0: a visual check, for example, with a boxplot can help to see, if outliers are present.

- 1: One way would be to remove outliers if their z-scores (after standardizing them) exceeds a certain threshold.

- 2: a common rule is the one that is also implemented in the boxplots. Outliers are those data point, that exceed the IQR multiplied by 1.5 in either direction.

- 3: another way would be to remove them by percentile, for example, by removind the top 5 and the bottom 5 percent of a variable (not recommended).

- 4: we can come up with our own rules, either theoretically or methodologically backed. In the fourth example, we exclude outliers based on a rule for which we used median and median absolute deviation instead of mean and standard deviation.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import median_abs_deviation

# --- 0: Visual Check for Outliers ---
sns.boxplot(y='lfstl_sle', data=data, color='lightblue', linewidth=1)
plt.title("Boxplot")
plt.ylabel("Sleeping Hours")
plt.show()

# --- 1: Z-Scores ---
mean = data['lfstl_sle'].mean()
std = data['lfstl_sle'].std()
z_scores = (data['lfstl_sle'] - mean) / std
data['lfstl_sle_outliers1'] = abs(z_scores) > 2

# --- 2: IQR Range ---
Q1 = data['lfstl_sle'].quantile(0.25)
Q3 = data['lfstl_sle'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
data['lfstl_sle_outliers2'] = (data['lfstl_sle'] < lower_bound) | (data['lfstl_sle'] > upper_bound)

# --- 3: Percentiles ---
lower_percentile = data['lfstl_sle'].quantile(0.05)
upper_percentile = data['lfstl_sle'].quantile(0.95)
data['lfstl_sle_outliers3'] = (data['lfstl_sle'] < lower_percentile) | (data['lfstl_sle'] > upper_percentile)

# --- 4: Modified Z-Scores ---
median = data['lfstl_sle'].median()
mad = median_abs_deviation(data['lfstl_sle'])
modified_z = 0.6745 * (data['lfstl_sle'] - median) / mad
data['lfstl_sle_outliers4'] = abs(modified_z) > 3.5

# --- Follow-up: Analyzing Outliers ---
outlier_data = data[data['lfstl_sle_outliers1'] == True]

For more complex data structures and intertwined variables, exploratory analyses can become a big part of the analysis. Some techniques help detect underlying connections between variables that help to make informed decisions for follow up analyses. The interpretation of these techniques can be difficult. Among these are, for example:

- 1: Principal Component Analysis (PCA): PCA reduces the dimensionality of data by transforming it into a set of uncorrelated principal components that capture the most variance. This helps to simplify complex datasets and to identify key structures.

- 2: K-means clustering: It groups data points into a number of clusters based on their similarity, where each point belongs to the cluster with the nearest mean. This helps uncover natural groupings or segments in unlabeled data.

- 3: Exploratory Factor Analysis (EFA): EFA identifies underlying latent factors that explain the correlations among observed variables. This helps reveal the hidden structure or dimensions in the data.

- 4: Latent Class Analysis (LCA): LCA identifies unobserved subgroups in a population based on patterns in observed variables. It is useful for discovering hidden distinct groups in the data and classifying individuals into these distinct groups.

For our purposes, we do not need to learn about PCA, k-means clustering, EFA, and LCA at this point; it might come up at a later point again.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from factor_analyzer import FactorAnalyzer
from sklearn.mixture import GaussianMixture

# --- 1: PCA ---
# Scale the variables
scaler = StandardScaler()
data[['age_scaled', 'faminc_scaled', 'sleep_scaled']] = scaler.fit_transform(
    data[['age', 'family_income', 'lfstl_sle']]
)

# Perform PCA
pca = PCA()
pca.fit(data[['age_scaled', 'faminc_scaled', 'sleep_scaled']])
data['out1_pc'] = pca.transform(data[['age_scaled', 'faminc_scaled', 'sleep_scaled']])[:, 0]

# --- 2: K-Means Clustering ---
# Set seed for reproducibility
np.random.seed(123)

# Perform K-means clustering
kmeans = KMeans(n_clusters=4, n_init=25, random_state=123)
kmeans.fit(data[['age_scaled', 'faminc_scaled', 'sleep_scaled']])
data['clusters'] = kmeans.labels_

# Optional: method to find optimal k (will take some time..)
"""
inertia = []
for k in range(1, 11):
    kmeans_temp = KMeans(n_clusters=k, n_init=25, random_state=123)
    kmeans_temp.fit(data[['age_scaled', 'faminc_scaled', 'sleep_scaled']])
    inertia.append(kmeans_temp.inertia_)

plt.plot(range(1, 11), inertia, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()
"""

# Visualize clusters (2D PCA for plotting)
pca_2d = PCA(n_components=2)
reduced_data = pca_2d.fit_transform(data[['age_scaled', 'faminc_scaled', 'sleep_scaled']])

plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=data['clusters'], cmap='viridis', alpha=0.6)
plt.title('K-Means Clustering')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.colorbar(label='Cluster')
plt.show()

# --- 3: EFA (Exploratory Factor Analysis) ---
# install factor_analyzer first: pip install factor-analyzer
X = data[['age_scaled', 'faminc_scaled', 'sleep_scaled']]
fa = FactorAnalyzer(n_factors=2, rotation='varimax')
fa.fit(X)
print("Factor Loadings:")
print(fa.loadings_)
print("\nFactor Variance:")
print(fa.get_factor_variance())


# --- 4: LCA (Latent Class Analysis) ---
# Convert categorical variables to numeric codes (one-hot encoding)
X_lca = pd.get_dummies(data[['lfstl_vap', 'lfstl_smo', 'lfstl_alc']])

# Perform LCA using Gaussian Mixture Models
np.random.seed(123)
lca = GaussianMixture(n_components=3, random_state=123, n_init=5)
lca.fit(X_lca)
data['lca_classes'] = lca.predict(X_lca)

Exploratory data analysis is important to help make us better decision for follow-up inferential analyses, and should not be skipped. Data used in social science research is generally messy and has a lot of noise (outliers, strange distributions, missings), and we should take the exploratory part serious. For many analyses, this is, however, only the first part.